# 📊 Notebook 02 — Statistická analýza & EKC
## Kombinovaný styl: Ukázka vzoru → Tvůj úkol

---
- 🔵 **[UKÁZKA]** — kompletní kód, prostuduj a spusť
- 🟡 **[TEĎ TY]** — napiš analogický kód sama

---
## ⚙️ Setup

Tuto buňku spusť vždy jako první — načte data a nastaví prostředí.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os, warnings
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)

df = pd.read_csv('../output/ekc_analysis.csv')
if 'log_gdp' not in df.columns:
    df['log_gdp'] = np.log(df['mean_gdp'])
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
df['income_group'] = pd.Categorical(df['income_group'], categories=income_order, ordered=True)
print(f'Načteno {len(df)} zemí. Sloupce: {df.columns.tolist()}')
assert 180 <= len(df) <= 210, f"Neocekavany pocet zemi: {len(df)} (ocekavano ~199) — zkontroluj notebook 01!"

---
## Sekce 1: Scatter plot HDP vs. lesní pokryv (= Úloha 1)

### 🟡 Scatter plot: `log_gdp` vs. `forest_change` (EKC pohled)

> ℹ️ **Kód je připraven předem** — tvorba vizualizací v Pythonu není cílem tohoto cvičení. Spusť buňku níže a porovnej výsledek s UKÁZKOU výše.

**Co očekávat**: Graf kde vidíš, že chudší země jsou spíše pod y=0 (odlesňování) a bohatší nad y=0 (zalesňování). Tento vizuální vzor je základ EKC hypotézy.

In [ ]:
# Grafická kontrola — kód je připraven předem
# Spusť tuto buňku a porovnej výsledek s UKÁZKOU výše
colors = {'Low income': '#d62728', 'Lower middle income': '#ff7f0e',
          'Upper middle income': '#1f77b4', 'High income': '#2ca02c'}

fig, ax = plt.subplots(figsize=(10, 6))
for group in income_order:
    mask = df['income_group'] == group
    ax.scatter(df[mask]['log_gdp'], df[mask]['forest_change'],
               label=group, color=colors[group], alpha=0.7, s=40)

ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8, label='0 — hranice zalesňování')
ax.set_xlabel('log(HDP per capita)')
ax.set_ylabel('Změna lesního pokryvu 1990–2025 [pp]')
ax.set_title('EKC scatter plot: HDP vs. změna lesa (199 zemí)')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

# Kontrolní výpis
above_zero = (df['forest_change'] > 0).sum()
below_zero = (df['forest_change'] < 0).sum()
print(f'Bodů nad y=0 (zalesňování): {above_zero}')
print(f'Bodů pod y=0 (odlesňování): {below_zero}')

---
## Sekce 2: Pearsonova korelace (= Úloha 2)

> **Proč Pearsonova korelace?**
> Pearsonův koeficient r měří sílu a směr **lineárního vztahu** mezi dvěma číselnými proměnnými. Pracujeme se dvěma číselnými sloupci (`log_gdp`, `forest_change`) — Pearson je pro ně přirozená první volba.
>
> **Pozor na omezení**: Pearson zachycuje pouze lineární vztah. EKC teorie ale předpovídá vztah nelineární (obrácenou parabolu) — proto bude hodnota r zdánlivě slabá, i když EKC platí. Plný obraz přidá kvadratická regrese v Sekci 3.

### 🔵 [UKÁZKA] Korelace: `log_gdp` vs. `forest_1990`

Ukažeme jak vypočítat Pearsonovu korelaci a interpretovat výsledek.

In [ ]:
# Vzor: Pearsonova korelace mezi dvěma proměnnými
# stats.pearsonr vrátí (r, p_value)
# r ∈ [-1, 1]: blízko 1 = silná kladná korelace, blízko -1 = silná záporná
# p < 0.05 → výsledek je statisticky významný

data_1 = df[['log_gdp', 'forest_1990']].dropna()
r_demo, p_demo = stats.pearsonr(data_1['log_gdp'], data_1['forest_1990'])

print('Ukázková korelace: log(HDP) vs. % lesa v roce 1990')
print(f'  r = {r_demo:.4f}')
print(f'  p = {p_demo:.4f}')
print(f'  Statisticky významná: {"ANO" if p_demo < 0.05 else "NE"}')
if r_demo > 0:
    print(f'  Interpretace: Bohatší země měly v roce 1990 VÍCE lesa (r > 0)')
else:
    print(f'  Interpretace: Bohatší země měly v roce 1990 MÉNĚ lesa (r < 0)')

### 🟡 [TEĎ TY] Korelace: `log_gdp` vs. `forest_change`

**Zadání**: Teď spočítej Pearsonovu korelaci mezi **log(HDP)** a **změnou lesa 1990–2025**.
Toto je klíčová korelace pro Q1 analýzu!

1. Vytvoř DataFrame `cor_data` jen s `log_gdp` a `forest_change` (bez NaN)
2. Spočítej korelaci `stats.pearsonr()`
3. Vypiš r, p-hodnotu a interpretaci
4. **Je korelace statisticky významná?**


**Očekávaný výstup (formát):**
```
Pearsonova korelace: r = 0.3xxx
p-hodnota: 0.00xx
Korelace je statisticky VYZNAMNA (p < 0.05)
```

In [ ]:
# TVŮJ KÓD ZDE
# cor_data = df[['log_gdp', 'forest_change']].dropna()
# r, p_value = stats.pearsonr(cor_data['log_gdp'], cor_data['forest_change'])
# print(f'Pearsonova korelace: r = {r:.4f}')
# print(f'p-hodnota: {p_value:.4f}')
# print(f'Korelace je statisticky {"VYZNAMNA" if p_value < 0.05 else "NEVYZNAMNA"} (p < 0.05)')

> 📝 **Jak číst výsledek:**
> - **Směr** (`r` kladné/záporné): kladné = bohatší země → více lesa; záporné = opak
> - **Síla** (`|r|`): méně než 0.3 = slabá, 0.3–0.7 = střední, nad 0.7 = silná
> - **Průkaznost** (`p < 0.05`): vztah není náhodný, platí statisticky
> - **Zamysli se**: Proč může být korelace slabá, i když EKC platí? (Nápověda: EKC je **nelineární**!)

---
## Sekce 3: Lineární vs. Kvadratická regrese (= Úloha 3)

> **Proč kvadratická a ne lineární regrese?**
> EKC teorie předpovídá konkrétní tvar vztahu: nejprve zhoršení životního prostředí s růstem HDP, pak zlepšení — tedy **obrat**. Přímka (lineární regrese) obrat zachytit neumí, protože vždy jen roste nebo klesá.
>
> Kvadratická funkce `y = ax² + bx + c` bod zlomu mít může: pokud vyjde `a < 0`, křivka má tvar ∩ — přesně jak EKC říká. Vyšší stupně polynomu (deg=3, 4...) by data sice lépe opisovaly, ale ztratily by ekonomický smysl — EKC předpovídá **jeden** bod obratu, ne více.

### 🔵 [UKÁZKA] Lineární regrese: log_gdp → forest_1990

In [ ]:
# Vzor: np.polyfit pro polynomiální regresi
# np.polyfit(x, y, deg) → vrátí koeficienty [a, b, ...] od nejvyššího stupně
# deg=1 → lineární (y = a·x + b)
# deg=2 → kvadratická (y = a·x² + b·x + c)

x_demo = data_1['log_gdp'].values
y_demo = data_1['forest_1990'].values

coeffs_lin_demo = np.polyfit(x_demo, y_demo, 1)  # Lineární
print(f'Lineární regrese (forest_1990 vs log_gdp):')
print(f'  y = {coeffs_lin_demo[0]:.3f}·x + {coeffs_lin_demo[1]:.3f}')

# Predikované hodnoty
y_pred_demo = np.polyval(coeffs_lin_demo, x_demo)

# R²
ss_res = np.sum((y_demo - y_pred_demo)**2)
ss_tot = np.sum((y_demo - y_demo.mean())**2)
r2_demo = 1 - ss_res/ss_tot
print(f'  R² = {r2_demo:.4f} (model vysvětluje {r2_demo*100:.1f}% variance)')

# Vizualizace
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_demo, y_demo, alpha=0.5, s=30, label='Pozorování')
x_line = np.linspace(x_demo.min(), x_demo.max(), 100)
ax.plot(x_line, np.polyval(coeffs_lin_demo, x_line), 'r-', label=f'Lineární (R²={r2_demo:.3f})')
ax.set_xlabel('log(HDP)')
ax.set_ylabel('% lesa v roce 1990')
ax.set_title('Ukázka: Lineární regrese')
ax.legend()
plt.tight_layout()
plt.show()

### 🟡 [TEĎ TY] Kvadratická regrese (EKC): log_gdp → forest_change

**Zadání**: Napiš **kvadratickou** (deg=2) regresi pro klíčová data projektu:
- X = `log_gdp`
- Y = `forest_change`

1. Připrav `x` a `y` (bez NaN)
2. Fit kvadratickým polynomem: `np.polyfit(x, y, 2)` → `coeffs_quad`
3. Výpočet R²
4. Výpočet bodu zlomu (vertex): `x_vertex = -b / (2*a)`, `gdp_inflection = np.exp(x_vertex)`
5. Spusť kontrolní vizualizaci z buňky níže — EKC křivka je připravena předem

> ✅ **Sanity check po výpočtu**:
> - Koeficient `a` by měl být **záporný** → křivka má tvar ∩ (to je EKC). Pokud `a > 0`, křivka je ∪ — zkontroluj, že `x = log_gdp` (ne samotné GDP bez logaritmu).
> - Bod zlomu by měl být přibližně **$5 000–$500 000/os.** Výsledek mimo tento rozsah = chyba ve výpočtu.

<details>
<summary>💡 Proč -b/(2a)?</summary>

Kvadratická funkce `y = ax² + bx + c` má bod zlomu tam, kde derivace `2ax + b = 0`, tedy `x = -b/(2a)`. Přes `np.exp()` převedeš `log(HDP)` zpět na USD — a dostaneš bod zlomu jako HDP na obyvatele.

</details>

**Očekávaný výstup (formát):**
```
Koeficienty: a=-0.36252, b=7.91240, c=-41.49626
Bod zlomu:   $54,885/os.
R²:          0.1456
```

In [ ]:
# TVŮJ KÓD ZDE
# reg_data = df[['log_gdp', 'forest_change']].dropna()
# x = reg_data['log_gdp'].values
# y = reg_data['forest_change'].values

# coeffs_quad = np.polyfit(x, y, 2)
# a, b, c = coeffs_quad

# x_vertex = -b / (2 * a)
# gdp_inflection = np.exp(x_vertex)

# y_pred = np.polyval(coeffs_quad, x)
# ss_res = np.sum((y - y_pred) ** 2)
# ss_tot = np.sum((y - y.mean()) ** 2)
# r_squared = 1 - ss_res / ss_tot

# print(f'Koeficienty: a={a:.5f}, b={b:.5f}, c={c:.5f}')
# print(f'Bod zlomu:   ${gdp_inflection:,.0f}/os.')
# print(f'R²:          {r_squared:.4f}')

In [ ]:
# Grafická kontrola — EKC křivka (kód připraven předem)
# Spusť tuto buňku až po dokončení kroků 1–4 výše
# (musí existovat: coeffs_quad, x_vertex, gdp_inflection)

if 'coeffs_quad' not in dir():
    print('⚠️  Nejdřív dokonči předchozí buňku — coeffs_quad není definováno!')
else:
    reg_data_plot = df[['log_gdp', 'forest_change']].dropna()
    x_plot = reg_data_plot['log_gdp'].values

    fig, ax = plt.subplots(figsize=(10, 6))

    for group in income_order:
        mask = df['income_group'] == group
        ax.scatter(df[mask]['log_gdp'], df[mask]['forest_change'],
                   label=group, color=colors[group], alpha=0.6, s=30)

    x_line = np.linspace(x_plot.min(), x_plot.max(), 200)
    y_line = np.polyval(coeffs_quad, x_line)
    ax.plot(x_line, y_line, 'k-', linewidth=2, label='EKC křivka (deg=2)')

    ax.axvline(x=x_vertex, color='purple', linestyle=':', linewidth=1.5,
               label=f'Bod zlomu: ${gdp_inflection:,.0f}/os.')
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

    ax.set_xlabel('log(HDP per capita)')
    ax.set_ylabel('Změna lesního pokryvu 1990–2025 [pp]')
    ax.set_title('EKC model: Kvadratická regrese s bodem zlomu')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

> 📝 **Jak číst výsledek:**
> - **Znaménko `a`** (první koeficient): záporné = křivka ve tvaru ∩ (EKC teorie potvrzena)
> - **Bod zlomu v USD**: při jakém HDP se trend obrací? Je to hodnota dostupná pro chudé země?
> - **R²**: kolik procent variance model vysvětluje? Kvadratický model lepší než lineární?
> - **Závěr**: Leží většina zemí pod nebo nad bodem zlomu?

> 📊 **Power BI — vizualizace 1 (Q1 — EKC scatter)**: Bodový graf (Scatter) → osa X: `log_gdp`, osa Y: `forest_change`, barvy: `income_group`. Zdroj: `ekc_analysis.csv`. Viz **PowerBI_pruvodce.md, část 4**.
>
> 📊 **Power BI — vizualizace 2 (Q1 — příjmové skupiny)**: Skupinový sloupcový graf → osa X: `income_group`, osa Y: průměr `forest_change`. Zdroj: `ekc_analysis.csv`. Viz **PowerBI_pruvodce.md, část 4**.

---
## Sekce 4: Test hypotézy — porovnání dvou skupin (= Úloha 4)

> **Proč Mann-Whitney U test a ne dvouvýběrový t-test?**
> Porovnáváme dvě skupiny zemí — ale `forest_change` není normálně rozděleno: většina zemí se pohybuje blízko nuly, ale některé mají extrémní hodnoty (−35 % nebo +8 %). Dvouvýběrový t-test předpokládá normální rozdělení v obou skupinách — při jeho porušení může být výsledek nespolehlivý.
>
> Mann-Whitney U test je **neparametrická alternativa t-testu**: místo průměrů porovnává pořadí (ranky) hodnot. Nepředpokládá normální rozdělení a je odolný vůči odlehlým hodnotám. Proto je pro naše data vhodnější volbou.
>
> **Proč k tomu přidáváme jednovýběrový t-test (bonus)?**
> Doplňkový t-test řeší jinou otázku: *Je průměrná změna lesa High income zemí kladná?* To je srovnání s pevnou referenční hodnotou 0 (= žádná změna), ne srovnání dvou skupin. Pro tento typ testování je `ttest_1samp` standardní volbou.

### 🔵 [UKÁZKA] Mann-Whitney U test: Low income vs. Lower middle income

In [ ]:
# Vzor: Mann-Whitney U test — neparametrický test pro porovnání distribucí
# alternative='greater' → jednostranný test: první skupina > druhá
# alternative='two-sided' → oboustranný test (jakýkoli rozdíl)

low = df[df['income_group'] == 'Low income']['forest_change'].dropna()
lower_mid = df[df['income_group'] == 'Lower middle income']['forest_change'].dropna()

print(f'Low income:           průměr={low.mean():.3f}, n={len(low)}')
print(f'Lower middle income:  průměr={lower_mid.mean():.3f}, n={len(lower_mid)}')

u_stat_demo, p_demo2 = stats.mannwhitneyu(lower_mid, low, alternative='greater')
print(f'\nMann-Whitney U test: Lower middle > Low income')
print(f'  U = {u_stat_demo:.0f}, p = {p_demo2:.4f}')
print(f'  Výsledek: {"Potvrzeno" if p_demo2 < 0.05 else "Nepotvrzeno"}')

### 🟡 [TEĎ TY] Klíčový test Q1: High income vs. Low income

**Zadání**: Otestuj hlavní hypotézu projektu (Q1):
- H1: **High income** země mají statisticky vyšší změnu lesa než **Low income** země

1. Vyfiltruj `forest_change` pro `High income` a `Low income`
2. Vypiš průměr a počet zemí v obou skupinách — pomůže to při interpretaci výsledku
3. Spusť `stats.mannwhitneyu(high, low, alternative='greater')`
4. Interpretuj výsledek
5. **Bonus**: Spusť jednovýběrový t-test: Je průměrná změna lesa High income > 0?
   → `stats.ttest_1samp(high_income, popmean=0, alternative='greater')`

<details>
<summary>💡 Proč alternative='greater'?</summary>

Říkáme tím, že testujeme **jednostrannou** hypotézu: zajímá nás jen jestli High income je *větší* než Low income. Přesně odpovídá naší H1. Dvoustranný test (`'two-sided'`) by byl konzervativnější a hůře by odhalil tento konkrétní směrový vztah.

</details>

**Očekávaný výstup (formát):**
```
High income: prumer=x.xxx, n=75
Low income:  prumer=-x.xxx, n=25
Mann-Whitney U test: High income > Low income
  U = xxxxx, p = 0.000x
  Vysledek: POTVRZENO
```

In [ ]:
# TVŮJ KÓD ZDE

# high = df[df['income_group'] == 'High income']['forest_change'].dropna()
# low  = df[df['income_group'] == 'Low income']['forest_change'].dropna()
# print(f'High income: prumer={high.mean():.3f}, n={len(high)}')
# print(f'Low income:  prumer={low.mean():.3f}, n={len(low)}')

# u_stat, p_mw = stats.mannwhitneyu(high, low, alternative='greater')
# print(f'\nMann-Whitney U test: High income > Low income')
# print(f'  U = {u_stat:.0f}, p = {p_mw:.4f}')
# print(f'  Výsledek: {"POTVRZENO" if p_mw < 0.05 else "Nepotvrzeno"}')

## Bonus: t-test (je průměr High income > 0?)
# t_stat, p_t = stats.ttest_1samp(high, popmean=0, alternative='greater')
# print(f'\nt-test (High income > 0): t={t_stat:.4f}, p={p_t:.4f}')

> 📝 **Jak číst výsledek:**
> - **Jednostranný test** (`alternative='greater'`): testujeme konkrétní směr — H1: High > Low
> - **p < 0.05** → H1 potvrzena: High income země skutečně zalesňují více
> - **p >= 0.05** → H0 nezamítnuta: nemáme dostatečný důkaz pro H1
> - **Závěr pro Q1**: Potvrzuje výsledek EKC hypotézu o vlivu bohatství na lesy?

---
## Sekce 5: Porovnání více skupin — Kruskal-Wallis (= Úloha 5)

> **Proč Kruskal-Wallis a ne jednocestná ANOVA?**
> Porovnáváme 7 regionů — pro více než dvě skupiny je jednocestná ANOVA klasická volba. Jenže ANOVA vyžaduje normální rozdělení v každé skupině a srovnatelné rozptyly. U nás jsou skupiny nestejně velké (North America = 3 země, Europe & Central Asia = 55 zemí) a data jsou zešikmená.
>
> Kruskal-Wallis je **neparametrická alternativa jednocestné ANOVA**. Stejně jako Mann-Whitney pracuje s pořadím hodnot a nevyžaduje normální rozdělení.
>
> **Důležité omezení**: Test odhalí, zda se aspoň jeden region statisticky liší, ale **neidentifikuje které** dvojice regionů se liší. K tomu by bylo potřeba post-hoc testování — to přesahuje rámec tohoto projektu.

### 🔵 [UKÁZKA] Kruskal-Wallis test napříč příjmovými skupinami

In [ ]:
# Vzor: Kruskal-Wallis — neparametrický test pro VÍCE než 2 skupiny
# Testujeme: Jsou všechny skupiny stejné, nebo se aspoň jedna liší?

income_groups_data = [
    df[df['income_group'] == g]['forest_change'].dropna().values
    for g in income_order
    if g in df['income_group'].values
]

h_stat_demo, p_kw_demo = stats.kruskal(*income_groups_data)

print(f'Kruskal-Wallis test: Jsou rozdíly mezi příjmovými skupinami statisticky významné?')
print(f'  H = {h_stat_demo:.4f}')
print(f'  p = {p_kw_demo:.6f}')
print(f'  Výsledek: {"STATISTICKY VÝZNAMNÉ" if p_kw_demo < 0.05 else "NEVÝZNAMNÉ"}')

### 🟡 [TEĎ TY] Regionální statistiky a Kruskal-Wallis test

**Zadání**: Spusť stejný test pro **regiony** (místo příjmových skupin).
Tím odpovíš na výzkumnou otázku Q2: Jsou regionální rozdíly statisticky průkazné?

1. Spočítej `groupby('region')` statistiky pro `forest_change`: n, mean_change, median_change, std_change → ulož jako `regional_summary`
2. Seřaď výsledek podle `mean_change` vzestupně, zaokrouhli na 3 desetinná místa
3. Vytvoř list polí `forest_change` pro každý region
4. Odstraň prázdné skupiny: `region_groups = [g for g in region_groups if len(g) > 0]`
5. Spusť `stats.kruskal(*region_groups)`
6. Interpretuj výsledek

**Očekávaný výstup (formát):**
```
                             n  mean_change  ...
region
Sub-Saharan Africa          46       -5.775  ...
Latin America & Caribbean   35       -3.204  ...
...
Europe & Central Asia       55        2.747  ...

Kruskal-Wallis test: Jsou regionální rozdíly statisticky průkazné?
  H = xx.xxxx
  p = 0.000xxx
  Vysledek: ANO - regionalni rozdily jsou statisticky VYZNAMNE
```

In [ ]:
# TVŮJ KÓD ZDE

## Krok 1-2: Skupinová statistika podle regionu
# regional_summary = (
#     df.groupby('region')['forest_change']
#     .agg(n='count', mean_change='mean', median_change='median', std_change='std')
#     .sort_values('mean_change')
#     .round(3)
# )
# print(regional_summary)

## Krok 3-5: Kruskal-Wallis test
# region_groups = [
#     df[df['region'] == r]['forest_change'].dropna().values
#     for r in df['region'].dropna().unique()
# ]
# region_groups = [g for g in region_groups if len(g) > 0]
# h_stat, p_kw = stats.kruskal(*region_groups)
# print(f'\nKruskal-Wallis test: Jsou regionální rozdíly statisticky průkazné?')
# print(f'  H = {h_stat:.4f}')
# print(f'  p = {p_kw:.6f}')
# print(f'  Výsledek: {"ANO - regionální rozdíly jsou statisticky VÝZNAMNÉ" if p_kw < 0.05 else "NEVÝZNAMNÉ"}')

> 📝 **Jak číst výsledek:**
> - Kruskal-Wallis testuje, zda **aspoň jeden** region se liší od ostatních
> - Statisticky významný výsledek = rozdíly jsou příliš velké na to, aby byly náhodné
> - Test **neřekne** které regiony se liší — to by vyžadovalo post-hoc analýzu
> - **Závěr pro Q2**: Závisí změna lesa na geografické poloze nad rámec ekonomiky?

> 📊 **Power BI — vizualizace 3 (Q2 — regiony)**: Sloupcový graf (vodorovný) → osa Y: `region`, osa X: `mean_change`, barvy: `region`. Zdroj: `regional_summary.csv`. Viz **PowerBI_pruvodce.md, část 4**.

---
## Sekce 6: Kvartily a outliers — Paradoxní země (= Úloha 6)

### 🔵 [UKÁZKA] Kvartilová analýza: forest_change

In [ ]:
# Vzor: Výpočet kvartilů a identifikace outlierů metodou IQR
# Kvartily rozdělí data na 4 stejné části
# Q1 = 25% hodnot je pod touto hranicí
# Q3 = 75% hodnot je pod touto hranicí
# IQR = Q3 - Q1 (mezikvartiové rozpětí)

q1 = df['forest_change'].quantile(0.25)
q2 = df['forest_change'].quantile(0.50)  # Medián
q3 = df['forest_change'].quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr  # Dolní hranice outlierů
upper_fence = q3 + 1.5 * iqr  # Horní hranice outlierů

outliers_iqr = df[(df['forest_change'] < lower_fence) | (df['forest_change'] > upper_fence)]

print(f'Kvartily forest_change:')
print(f'  Q1 (25%):  {q1:.3f}%')
print(f'  Q2 (50%):  {q2:.3f}%  (medián)')
print(f'  Q3 (75%):  {q3:.3f}%')
print(f'  IQR:       {iqr:.3f}')
print(f'  Outliery (IQR metoda): {len(outliers_iqr)} zemí')

### 🟡 [TEĎ TY] Paradoxní země (Q3 analýza)

> ⚠️ **Nový přístup**: Ukázka výše hledala outliery v `forest_change` pomocí IQR metody.
> Tady děláme **jinou analýzu** — hledáme ekonomicky paradoxní země pomocí **peer-group residuálu**:
> „O kolik se tato země liší od mediánu zemí na stejné ekonomické úrovni?"
> Teprve velká odchylka (≥ 1 SD) je skutečné překvapení.
>
> **Proč ne prostý práh `forest_change < 0`?** Norsko (−0.3 %), Belgie (−0.2 %) nebo Austrálie (−0.1 %) by byly „bohaté odlesňovatelé" — ale tyto malé záporné hodnoty jsou normální variací okolo High income mediánu +1.21 pp. Nejsou paradoxem.

**Krok 0 — Geografický filtr**: Nejprve vyloučíme `forest_1990 < 2.0 %` — pouštní státy (Kuvajt, Katar, Libye...), jejichž pokryv je omezen fyzikální geografií.

**Zadání** (kroky 1–5):
1. Vytvoř `df_e` = způsobilé země (`forest_1990 >= 2.0`), vypiš počet vyloučených
2. Vypočítej `income_medians` = mediány `forest_change` pro každou příjmovou skupinu z `df_e`
3. Přidej do `df` sloupec `peer_residual`:
   - způsobilé: `forest_change − income_medians[income_group]`
   - vyloučené: `NaN`
4. Vypočítej `residual_std` — prahová hodnota je **±1 SD**
5. Identifikuj outliery a přidej `outlier_category`:
   - `'rich_deforester'`: High income + peer_residual < −1 SD + forest_change < 0
   - `'poor_reforester'`: Low/Lower middle income + peer_residual > +1 SD + forest_change > 0
   - `'expected_positive'` / `'expected_negative'`: ostatní dle směru
   - `'geographic_excluded'`: NaN peer_residual

> 💡 Krok 2: `income_medians = df_e.groupby('income_group')['forest_change'].median()`
>
> 💡 Krok 3: `df.apply(lambda r: r['forest_change'] - income_medians.get(r['income_group'], float('nan')) if r['forest_1990'] >= 2.0 else float('nan'), axis=1)`

> ✅ **Sanity check**: SD residuálů ≈ 7.06 pp; High income medián = +1.21 pp; Low income = −4.38 pp

**Očekávaný výstup (formát):**
```
Geografický filtr: 24 zemí vyloučeno (forest_1990 < 2.0 %)
Analyzovaných zemí: 175

Mediány forest_change podle příjmové skupiny:
  Low income:             -4.38 pp
  Lower middle income:    -2.83 pp
  Upper middle income:     0.00 pp
  High income:            +1.21 pp

Residual std = 7.06 pp  →  práh ±1 SD = ±7.06 pp

Bohaté odlesňovatelé (3):
       Seychelles  High income  forest_change: -16.55 %  peer_residual: -17.76 pp
   American Samoa  High income  forest_change: -11.10 %  peer_residual: -12.31 pp
           Brunei  High income  forest_change:  -6.26 %  peer_residual:  -7.47 pp

Chudé zalesňovatelé (5):
       Vietnam  Lower middle income  forest_change: +17.28 %  peer_residual: +20.11 pp
        Rwanda  Low income           forest_change:  +8.01 %  peer_residual: +12.39 pp
         Ghana  Lower middle income  forest_change:  +5.49 %  peer_residual:  +8.32 pp
        Bhutan  Lower middle income  forest_change:  +4.41 %  peer_residual:  +7.24 pp
    Cape Verde  Lower middle income  forest_change:  +4.36 %  peer_residual:  +7.19 pp
```

In [ ]:
# TVŮJ KÓD ZDE

## Krok 0: Geografický filtr
# FOREST_MIN = 2.0
# n_excluded = (df['forest_1990'] < FOREST_MIN).sum()
# print(f'Geografický filtr: {n_excluded} zemí vyloučeno (forest_1990 < {FOREST_MIN} %)')
# eligible = df['forest_1990'] >= FOREST_MIN
# df_e = df[eligible].copy()
# print(f'Analyzovaných zemí: {len(df_e)}')

## Krok 1: Mediány příjmových skupin (peer skupiny)
# income_medians = df_e.groupby('income_group')['forest_change'].median()
# print('\nMediány forest_change podle příjmové skupiny:')
# for grp, med in income_medians.items():
#     print(f'  {grp:<27}: {med:+.2f} pp')

## Krok 2: Peer residuál (NaN pro vyloučené země)
# df['peer_residual'] = df.apply(
#     lambda r: r['forest_change'] - income_medians.get(r['income_group'], float('nan'))
#               if r['forest_1990'] >= FOREST_MIN else float('nan'),
#     axis=1
# )
# residual_std = df['peer_residual'].dropna().std()
# print(f'\nResiduální std = {residual_std:.2f} pp  →  práh ±1 SD = ±{residual_std:.2f} pp')

## Krok 3: Identifikace outlierů
# rich_deforesters = df[eligible & (df['income_group'] == 'High income') &
#                       (df['peer_residual'] < -residual_std) & (df['forest_change'] < 0)]
# poor_reforesters = df[eligible & df['income_group'].isin(['Low income', 'Lower middle income']) &
#                       (df['peer_residual'] > residual_std) & (df['forest_change'] > 0)]

# print(f'\nBohaté odlesňovatelé ({len(rich_deforesters)}):')
# print(rich_deforesters[['country', 'income_group', 'region', 'mean_gdp', 'forest_change', 'peer_residual']]
#       .sort_values('peer_residual').to_string(index=False))
# print(f'\nChudé zalesňovatelé ({len(poor_reforesters)}):')
# print(poor_reforesters[['country', 'income_group', 'region', 'mean_gdp', 'forest_change', 'peer_residual']]
#       .sort_values('peer_residual', ascending=False).to_string(index=False))

## Krok 4: Klasifikační funkce a sloupec outlier_category
# def classify_outlier(row):
#     if pd.isna(row.get('peer_residual', float('nan'))):
#         return 'geographic_excluded'
#     if (str(row['income_group']) == 'High income' and
#             row['peer_residual'] < -residual_std and row['forest_change'] < 0):
#         return 'rich_deforester'
#     elif (str(row['income_group']) in ['Low income', 'Lower middle income'] and
#               row['peer_residual'] > residual_std and row['forest_change'] > 0):
#         return 'poor_reforester'
#     elif row['forest_change'] >= 0:
#         return 'expected_positive'
#     else:
#         return 'expected_negative'
# df['outlier_category'] = df.apply(classify_outlier, axis=1)

In [ ]:
# Grafická kontrola — EKC scatter s peer-group outliery (kód připraven předem)
# Spusť tuto buňku až po dokončení kroků 1–4 výše
# Vyžaduje: df['outlier_category'], df['peer_residual'], coeffs_quad (ze Sekce 3)

if 'outlier_category' not in df.columns:
    print('⚠️  Nejdřív dokonči předchozí buňku — outlier_category není definováno!')
elif 'coeffs_quad' not in dir():
    print('⚠️  Spusť nejdřív Sekci 3 — coeffs_quad není definováno!')
else:
    cat_styles = {
        'rich_deforester':    ('#d62728', 80, 0.9),
        'poor_reforester':    ('#2ca02c', 80, 0.9),
        'expected_positive':  ('#aec7e8', 20, 0.3),
        'expected_negative':  ('#ffbb78', 20, 0.3),
        'geographic_excluded':('#cccccc', 15, 0.25),
    }
    fig, ax = plt.subplots(figsize=(12, 7))
    for cat, (color, size, alpha) in cat_styles.items():
        mask = df['outlier_category'] == cat
        ax.scatter(df[mask]['log_gdp'], df[mask]['forest_change'],
                   color=color, s=size, alpha=alpha,
                   label=f'{cat.replace("_"," ").title()} (n={mask.sum()})', zorder=3)
    # EKC křivka
    x_line = np.linspace(df['log_gdp'].dropna().min(), df['log_gdp'].dropna().max(), 200)
    ax.plot(x_line, np.polyval(coeffs_quad, x_line), 'k-', linewidth=2, label='EKC křivka', zorder=4)
    # Popisky outlierů
    for _, row in df[df['outlier_category'].isin(['rich_deforester', 'poor_reforester'])].iterrows():
        if pd.notna(row['log_gdp']) and pd.notna(row['forest_change']):
            ax.annotate(row['country'], xy=(row['log_gdp'], row['forest_change']),
                        fontsize=7, ha='left', va='bottom')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_xlabel('log(HDP per capita)')
    ax.set_ylabel('Změna lesního pokryvu 1990–2025 [pp]')
    ax.set_title('EKC model: Peer-group outliery — vzdálenost od mediánu příjmové skupiny ≥ 1 SD')
    ax.legend(fontsize=8, loc='upper left')
    plt.tight_layout()
    plt.savefig('../output/q3_outliers_ekc.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('\nKontrolní souhrn outlier_category:')
    print(df['outlier_category'].value_counts())

> 📊 **Power BI — vizualizace 4 (Q3 — mapa světa)**: Choropleth mapa → umístění: `code`, barva: `forest_change`. Zdroj: `ekc_analysis.csv`. Viz **PowerBI_pruvodce.md, část 4**.
>
> 📊 **Power BI — vizualizace 5 (Q3 — paradoxní země)**: Tabulka → sloupce: `country`, `income_group`, `forest_change`, `outlier_category`, `policy_score`. Zdroj: `outliers_with_policy.csv`. Viz **PowerBI_pruvodce.md, část 4**.

---
## Sekce 7: Forest Policy (= Úloha 7)

> **Proč analyzujeme lesní politiku?**
> V Sekci 6 jsme identifikovali 5 chudých zemí, které zalesňují navzdory nízkému HDP. Co je za tím? Jedna z hypotéz: **institucionální kapacita** — země s národní politikou, legislativou a zapojenými stakeholdery mají nástroje k plánování a vymáhání dlouhodobých programů obnovy lesů i bez vysokého HDP.
>
> FAO sbírá pro každou zemi, zda má: (1) národní politiku pro udržitelnou správu lesů (SFM), (2) odpovídající legislativu, (3) platformu pro zapojení stakeholderů. Součet těchto tří binárních ukazatelů tvoří `policy_score` (0–3) — proxy pro sílu lesního gouvernancu.
>
> **Testovaná hypotéza**: Chudé zalesňovatelé mají vyšší `policy_score` než odpovídá jejich ekonomické úrovni — tedy silnější instituce jsou spojena s lepšími výsledky i bez bohatství.

### 🔵 [UKÁZKA] Načtení a zpracování policy dat

In [ ]:
# Vzor: Načtení FAO Forest Policy dat a vytvoření policy score
fp_raw = pd.read_csv('../../data_raw/Forest_Policy_Legislation.csv', on_bad_lines='skip')
fp = fp_raw[['iso3', 'National policies supporting SFM',
             'National legislations supporting SFM',
             'National platform for stakeholder participation']].copy()
fp.columns = ['code', 'has_policy', 'has_legislation', 'has_platform']

# Konverze yes/no → True/False
for col in ['has_policy', 'has_legislation', 'has_platform']:
    fp[col] = fp[col].str.strip().str.lower().map({'yes': True, 'no': False})

# Policy score = počet zavedených opatření
fp['policy_score'] = fp[['has_policy', 'has_legislation', 'has_platform']].sum(axis=1)

print(f'Forest Policy data načtena: {len(fp)} zemí')
print(f'Policy score distribuce:')
print(fp['policy_score'].value_counts().sort_index())
print(f'\nPříklad:')
print(fp[fp['code'].isin(['CZE', 'ETH', 'BRA', 'DEU'])][['code', 'has_policy', 'has_legislation', 'policy_score']])

### 🟡 [TEĎ TY] Analýza: Mají paradoxní země jinou politiku?

**Zadání**: Propoj policy data s paradoxními zeměmi ze Sekce 6 (outlier_category):
1. `pd.merge(df[df['outlier_category'].isin(['rich_deforester', 'poor_reforester'])], fp, on='code', how='left')`
2. Porovnej průměrný `policy_score` pro `rich_deforester` vs. `poor_reforester`
3. Nakresli scatter plot: `policy_score` (x) vs. `forest_change` (y), barva podle kategorie
4. **Závěr**: Liší se lesní politika mezi skupinami? Koreluje silná politika s úspěchem chudých zemí?

**Očekávaný výstup (formát):**
```
                  n  mean  median
outlier_category
poor_reforester   5   3.0     3.0
rich_deforester   3  2.67     3.0
```

> 🔍 **Klíčový závěr**: Všechny chudé zalesňovatelé (Vietnam, Rwanda, Ghana, Bhutan, Cape Verde) mají **policy_score = 3 (maximum)**. Brunei je jediná paradoxní země s neúplnou politikou (score = 2). Globální průměr je 2.30 — paradoxní země skórují nadprůměrně.
>
> Silná lesní politika **koreluje** s úspěchem chudých zalesňovatelů. Ale korelace ≠ kauzalita — Rwanda, Vietnam i Bhutan mají i jiné specifika (poválečná obnova, hustota obyvatelstva, geografická izolace), která mohla hrát stejnou nebo větší roli.

In [ ]:
# TVŮJ KÓD ZDE

## Krok 1: Merge outliers s policy daty
# outlier_df = df[df['outlier_category'].isin(['rich_deforester', 'poor_reforester'])].copy()
# outlier_policy = pd.merge(outlier_df, fp, on='code', how='left')

## Krok 2: Průměrný policy_score podle kategorie
# policy_by_cat = outlier_policy.groupby('outlier_category')['policy_score'].agg(
#     n='count', mean='mean', median='median'
# ).round(2)
# print(policy_by_cat)

## Krok 3: Scatter plot policy_score vs. forest_change (barva podle outlier_category)
# scatter_colors = {'rich_deforester': '#d62728', 'poor_reforester': '#2ca02c'}
# fig, ax = plt.subplots(figsize=(8, 5))
# for cat, color in scatter_colors.items():
#     mask = outlier_policy['outlier_category'] == cat
#     ax.scatter(outlier_policy[mask]['policy_score'],
#                outlier_policy[mask]['forest_change'],
#                color=color, label=cat, s=60, alpha=0.8)
# ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
# ax.set_xlabel('Policy score (0–3)')
# ax.set_ylabel('Změna lesa 1990–2025 [pp]')
# ax.set_title('Paradoxní země: Lesní politika vs. změna lesa')
# ax.legend()
# plt.tight_layout()
# plt.show()

> 📊 **Power BI — vizualizace 6 (Q3 — peer_residual)**: Clustered bar chart (horizontální) → osa Y: `country`, osa X: `peer_residual`, barva: `outlier_category`. Zdroj: `outliers_with_policy.csv`. Viz **PowerBI_pruvodce.md, část 4**.
>
> 📊 **Power BI — vizualizace 7 (Q3 — lesní politika)**: Clustered bar chart → osa X: `outlier_category`, osa Y: průměr `policy_score`, referenční čára: 2.30. Zdroj: `outliers_with_policy.csv`. Viz **PowerBI_pruvodce.md, část 4**.

---
## Sekce 8: Export výstupů pro Power BI (= Úloha 8)

### 🔵 [UKÁZKA] Export regionálního souhrnu


In [ ]:
# Vzor: Export DataFrame do CSV
# regional_summary se zde přepočítává znovu — UKÁZKA musí fungovat
# nezávisle na TEĎ TY buňce ze Sekce 5 (student ji mohl přeskočit)
os.makedirs('../output', exist_ok=True)

regional_summary = (
    df.groupby('region')['forest_change']
    .agg(n='count', mean_change='mean', median_change='median', std_change='std')
    .sort_values('mean_change')
    .round(3)
    .reset_index()
)
regional_summary.to_csv('../output/regional_summary.csv', index=False, encoding='utf-8-sig')
print(f'✅ Uloženo regional_summary.csv ({len(regional_summary)} regionů)')

### 🟡 [TEĎ TY] Export ostatních souborů pro Power BI

**Zadání**: Exportuj zbývající soubory do `../output/`:
1. `ekc_analysis.csv` — hlavní dataset `df` (s outlier_category)
2. `ekc_regression_curve.csv` — 100 bodů pro EKC křivku
   - `x_line = np.linspace(df['log_gdp'].min(), df['log_gdp'].max(), 100)`
   - DataFrame se sloupci: `log_gdp_fit`, `gdp_fit`, `forest_pred`
   - Použij `coeffs_quad` z Sekce 3
3. `outliers.csv` — paradoxní země (kde `outlier_category` je 'rich_deforester' nebo 'poor_reforester')
4. `outliers_with_policy.csv` — paradoxní země + policy data (ze Sekce 7)
   - Použij `outlier_policy` ze Sekce 7, vyber sloupce:
     `country`, `income_group`, `region`, `mean_gdp`, `forest_change`,
     `peer_residual`, `outlier_category`, `has_policy`, `has_legislation`, `policy_score`

> ℹ️ Export vyžaduje proměnné z předchozích sekcí: `coeffs_quad` (Sekce 3), `outlier_policy` (Sekce 7) — spusť je nejdřív.

**Očekávaný výstup:**
```
✅ ekc_analysis.csv          (199 řádků)
✅ ekc_regression_curve.csv  (100 bodů pro EKC křivku)
✅ outliers.csv              (8 zemí — 3 rich_deforester + 5 poor_reforester)
✅ outliers_with_policy.csv
```

> ✅ **Ověření**: Otevři složku `../output/` v Průzkumníku souborů — všechny soubory musí existovat. Zkontroluj `ekc_regression_curve.csv` v Excelu — měla by mít přesně 100 řádků a sloupce `log_gdp_fit`, `gdp_fit`, `forest_pred`.

In [ ]:
# TVŮJ KÓD ZDE

# os.makedirs('../output', exist_ok=True)

# df.to_csv('../output/ekc_analysis.csv', index=False, encoding='utf-8-sig')
# print(f'✅ ekc_analysis.csv          ({len(df)} řádků)')

# x_line = np.linspace(df['log_gdp'].min(), df['log_gdp'].max(), 100)
# ekc_curve = pd.DataFrame({'log_gdp_fit': x_line, 'gdp_fit': np.exp(x_line),
#                            'forest_pred': np.polyval(coeffs_quad, x_line)})
# ekc_curve.to_csv('../output/ekc_regression_curve.csv', index=False, encoding='utf-8-sig')
# print('✅ ekc_regression_curve.csv  (100 bodů pro EKC křivku)')

# df[df['outlier_category'].isin(['rich_deforester', 'poor_reforester'])]\
#     .to_csv('../output/outliers.csv', index=False, encoding='utf-8-sig')
# print('✅ outliers.csv')

# cols = ['country', 'income_group', 'region', 'mean_gdp', 'forest_change',
#         'peer_residual', 'outlier_category', 'has_policy', 'has_legislation', 'policy_score']
# cols = [c for c in cols if c in outlier_policy.columns]
# outlier_policy[cols].to_csv('../output/outliers_with_policy.csv', index=False, encoding='utf-8-sig')
# print('✅ outliers_with_policy.csv')

---
## ✅ Hotovo?

Zkontroluj:
- [ ] Scatter plot HDP vs. změna lesa nakreslen (Sekce 1)
- [ ] Korelace spočítána a interpretována (Sekce 2)
- [ ] EKC křivka nakreslena, bod zlomu vypočítán (Sekce 3)
- [ ] Mann-Whitney test pro Q1 proveden (Sekce 4)
- [ ] Kruskal-Wallis test pro Q2 proveden (Sekce 5)
- [ ] Paradoxní země identifikovány, sloupec `outlier_category` přidán (Sekce 6)
- [ ] Forest Policy analýza provedena (Sekce 7)
- [ ] 5 souborů exportováno do `../output/` (Sekce 8)

**Pokračuj**: [`../PowerBI_pruvodce.md`](../PowerBI_pruvodce.md)
**Bonus**: `03_sql_bonus.ipynb`